In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from pathlib import Path
from comet_ml import Experiment

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import produce_na
from imputation import imputation, get_rmse

np.random.seed(42)

In [2]:
key = Path("../COMET_API_KEY.zshrc").read_text(encoding="utf-8").strip()
os.environ["COMET_API_KEY"] = key

experiment = Experiment(
    api_key=os.environ.get("COMET_API_KEY"),
    project_name="missing-data-imputation",
    workspace=None
)
experiment.set_name("Comparison_Run_v1")

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/da7d62c00c47410eb2bfb95d8cb10566



In [3]:
results_buffer = [] # список для хранения результатов, чтобы потом сохранить в эксель

# datasets = ["housing", "adult", "AirQuality"]
datasets = ["housing"]
# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": "population"
}
#  глобальный таргет
global_target = {
    "housing": "median_house_value"
}
ratios = [0.05, 0.20, 0.50]
# mechanisms = ["MCAR", "MAR", "MNAR"]
mechanisms = ["MCAR"]
# methods = ["Mean", "KNN", "MICE", "MissForest"]
methods = ["Simple", "KNN"]

EXPERIMENT_CONFIG = {
    "Simple": [
        {"strategy": "mean"},
        {"strategy": "median"},
    ],
    "KNN": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
}

# ЦИКЛ ПО ДАТАСЕТАМ
for dataset_name in datasets:
    file_path = Path("..") / f"data/processed/{dataset_name}/ground_truth.csv"
    df = pd.read_csv(file_path)
    # удаляем глобальный таргет, чтобы случайно не подглядеть
    df = df.drop(columns=[global_target[dataset_name]])
    # ЦИКЛ ПО МЕТОДАМ ВОЗНИКНОВЕНИЯ ПРОПУСКОВ
    for mechanism in mechanisms:
        # ЦИКЛ ПО КОЛИЧЕСТВУ ПРОПУСКОВ
        for ratio in ratios:
            # генерация пропусков
            df_incomplete, mask = produce_na(
                data=df, 
                target_col=[columns_target[dataset_name]], 
                mechanism=mechanism, 
                ratio=ratio)
            df_incomplete_numeric = df_incomplete.select_dtypes(include='number')
            # ЦИКЛ ПО МЕТОДАМ ВОССТАНОВЛЕНИЯ
            for method, param_list in EXPERIMENT_CONFIG.items():
                # ЦИКЛ ПО ГИПЕРПАРАМЕТРАМ
                for params in param_list:
                    params_str = "_".join([f"{k}={v}" for k, v in params.items()])
                    
                    start = time.perf_counter()
                    imputed_data = imputation(df_incomplete=df_incomplete_numeric,
                                            method=method,
                                            **params)
                    end = time.perf_counter()
                    rmse_score = get_rmse(df.loc[mask, columns_target[dataset_name]], imputed_data.loc[mask, columns_target[dataset_name]])
                    # model_accuracy = ... (метрика модели, если есть)
                    time_taken = end - start

                    # ЛОГИРОВАНИЕ
                    current_result = {
                        "dataset": dataset_name,
                        "mechanism": mechanism,
                        "ratio": ratio,
                        "method": method,
                        "params": params_str, # Сохраняем как строку для удобства
                        "rmse": rmse_score,
                        "time": time_taken,
                    }

                    results_buffer.append(current_result)

                    # Логируем параметры эксперимента
                    metric_name_unique = f"RMSE_{mechanism}_{method}_{params_str}"
                    
                    experiment.log_metric(
                        metric_name_unique, 
                        rmse_score, 
                        step=int(ratio * 100) # Ось X: 5 -> 20 -> 50
                    )
                    
                    # Логируем время так же
                    time_name_unique = f"Time_{mechanism}_{method}_{params_str}"
                    experiment.log_metric(time_name_unique, time_taken, step=int(ratio * 100))

                    print(f"Done: {dataset_name} | {mechanism} | {method} ({params_str}) | {ratio} | RMSE: {rmse_score:.4f}")

    df_results = pd.DataFrame(results_buffer)
    df_results.to_excel(f"results_backup_{dataset_name}.xlsx", index=False)


Done: housing | MCAR | Simple (strategy=mean) | 0.05 | RMSE: 1214.7930
Done: housing | MCAR | Simple (strategy=median) | 0.05 | RMSE: 1250.3822
Done: housing | MCAR | KNN (n_neighbors=3) | 0.05 | RMSE: 461.3830
Done: housing | MCAR | KNN (n_neighbors=5) | 0.05 | RMSE: 418.4203
Done: housing | MCAR | KNN (n_neighbors=7) | 0.05 | RMSE: 408.2554
Done: housing | MCAR | Simple (strategy=mean) | 0.2 | RMSE: 1194.9751
Done: housing | MCAR | Simple (strategy=median) | 0.2 | RMSE: 1221.0323
Done: housing | MCAR | KNN (n_neighbors=3) | 0.2 | RMSE: 477.1953
Done: housing | MCAR | KNN (n_neighbors=5) | 0.2 | RMSE: 439.2993
Done: housing | MCAR | KNN (n_neighbors=7) | 0.2 | RMSE: 457.3087
Done: housing | MCAR | Simple (strategy=mean) | 0.5 | RMSE: 1182.5674
Done: housing | MCAR | Simple (strategy=median) | 0.5 | RMSE: 1209.2429
Done: housing | MCAR | KNN (n_neighbors=3) | 0.5 | RMSE: 542.5934
Done: housing | MCAR | KNN (n_neighbors=5) | 0.5 | RMSE: 538.1082
Done: housing | MCAR | KNN (n_neighbors=7

In [4]:
experiment.log_table("final_results.csv", df_results)

experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : Comparison_Run_v1
COMET INFO:     url                   : https://www.comet.com/kam2607kam/missing-data-imputation/da7d62c00c47410eb2bfb95d8cb10566
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     RMSE_MCAR_KNN_n_neighbors=3 [3]      : (461.38301382906866, 542.5934441176887)
COMET INFO:     RMSE_MCAR_KNN_n_neighbors=5 [3]      : (418.42032913272476, 538.1082133290231)
COMET INFO:     RMSE_MCAR_KNN_n_neighbors=7 [3]      : (408.2554105232061, 532.0684467718091)
COMET INFO:     RMSE_MCAR_Simple_strategy=mean [3]   : (1182.5674193453801, 1214.7929651071477)
COMET INFO:     RMSE_MCAR_Simple_strategy=median [3] : (1209.2429482650455, 1250.3822342